In [ ]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Intro to Grounding with Gemini in Agent Platform (v2)

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fkazunori279%2Fgcp-blogs%2Fmain%2F20260602_gemini_grounding%2Fintro-grounding-gemini-v2.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/agent-platform/workbench/instances?download_url=https://raw.githubusercontent.com/kazunori279/gcp-blogs/main/20260602_gemini_grounding/intro-grounding-gemini-v2.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

<p>
<b>Share to:</b>

<a href="https://www.linkedin.com/sharing/share-offsite/?url=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/8/81/LinkedIn_icon.svg" alt="LinkedIn logo">
</a>

<a href="https://bsky.app/intent/compose?text=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/7/7a/Bluesky_Logo.svg" alt="Bluesky logo">
</a>

<a href="https://twitter.com/intent/tweet?url=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/5a/X_icon_2.svg" alt="X logo">
</a>

<a href="https://reddit.com/submit?url=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2.ipynb" target="_blank">
  <img width="20px" src="https://redditinc.com/hubfs/Reddit%20Inc/Brand/Reddit_Logo.png" alt="Reddit logo">
</a>

<a href="https://www.facebook.com/sharer/sharer.php?u=https%3A//github.com/kazunori279/gcp-blogs/blob/main/20260602_gemini_grounding/intro-grounding-gemini-v2.ipynb" target="_blank">
  <img width="20px" src="https://upload.wikimedia.org/wikipedia/commons/5/51/Facebook_f_logo_%282019%29.svg" alt="Facebook logo">
</a>
</p>

| Authors |
| --- |
| [Kazunori Sato](https://github.com/kazunori279) |

## Overview

[Gemini Grounding in Agent Platform](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/grounding/overview) lets you use generative text models to generate content grounded in your own documents and data. This capability lets the model access information at runtime that goes beyond its training data. By grounding model responses in [Google Search](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/grounding/grounding-with-google-search) results or your own data, LLMs that are grounded in data can produce more accurate, up-to-date, and relevant responses.

This notebook demonstrates **6 grounding techniques** used in the [Concierge AI](https://grounding-demo-761793285222.us-east1.run.app/) demo app — a [Gemini Live API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/live-api) voice agent that plays the role of a luxury hotel concierge, powered by [Google Search](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/grounding/grounding-with-google-search), [Google Maps](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/grounding/grounding-with-google-maps), and [Vector Search 2.0](https://docs.cloud.google.com/gemini-enterprise-agent-platform/build/vector-search-2/overview) grounding.

| # | Grounding Type | API / Feature | Concierge Tool |
|---|---|---|---|
| 1 | Google Search | Built-in `GoogleSearch` tool | `web_search` (text) |
| 2 | Google Maps | `GoogleMaps` tool (place search, route, search along route) | `web_search` (maps), `maps_route`, `maps_search_along_route` |
| 3 | **Vector Search 2.0** | `vectorsearch_v1beta` SDK (semantic search with auto-embeddings) | `vs2_search` |
| 4 | Image Search | `enhancedContent.imageSearch` via REST | `web_search` (images) |
| 5 | Video Search | Google Search with video-biased query | `web_search` (videos) |
| 6 | Image Search for Image Generation | `searchTypes.imageSearch` via REST | `request_postcard` → `illustrate_place` |

### Objective

In this tutorial, you learn how to:

- Generate LLM text grounded in [Google Search](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/grounding/grounding-with-google-search) results
- Generate LLM text grounded in Google Maps data (place search, route, search along route)
- Create a Vector Search 2.0 collection with auto-embeddings and run semantic search
- Search for images via `enhancedContent.imageSearch`
- Search for YouTube videos via Google Search grounding
- Generate images grounded in Google Image Search results via `searchTypes.imageSearch`

This tutorial uses the following Google Cloud AI services and resources:

- Agent Platform ([Gemini API](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models))
- Agent Platform Vector Search 2.0

The steps performed include:

- Configuring the LLM and prompt for various examples
- Sending example prompts with Google Search and Google Maps grounding
- Setting up a Vector Search 2.0 collection with auto-embeddings for a hotel knowledge base
- Searching for images and videos via REST API
- Generating images grounded in Google Image Search results

## Before you begin

### Set up your Google Cloud project

**The following steps are required, regardless of your notebook environment.**

1. [Select or create a Google Cloud project](https://console.cloud.google.com/cloud-resource-manager). When you first create an account, you get a $300 free credit towards your compute/storage costs.
1. [Make sure that billing is enabled for your project](https://cloud.google.com/billing/docs/how-to/modify-project).
1. Enable the [Agent Platform and Vector Search APIs](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com,vectorsearch.googleapis.com).
1. If you are running this notebook locally, you need to install the [Cloud SDK](https://cloud.google.com/sdk).

### Install [Google Gen AI SDK](https://googleapis.github.io/python-genai/), [ADK](https://google.github.io/adk-docs/), and Vector Search SDK

Install the following packages required to execute this notebook.

In [ ]:
# noqa: E225
%pip install --upgrade --quiet \
    google-adk \
    google-cloud-vectorsearch \
    google-auth \
    requests

### Authenticate your Google Cloud account

If you are running this notebook on Google Colab, you will need to authenticate your environment. To do this, run the new cell below. This step is not required if you are using Colab Enterprise or Agent Platform Workbench.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth
    auth.authenticate_user()

### Set Google Cloud project information and create client

To get started using Agent Platform, you must have an existing Google Cloud project and [enable the Agent Platform API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/start).

**If you don't know your project ID**, try the following:
* Run `gcloud config list`.
* Run `gcloud projects list`.
* See the support page: [Locate the project ID](https://support.google.com/googleapi/answer/7014113)

You can also change the `LOCATION` variable used by Agent Platform. Learn more about [Agent Platform regions](https://docs.cloud.google.com/gemini-enterprise-agent-platform/resources/locations).

In [ ]:
import os  # noqa: E402

PROJECT_ID = "[your-project-id]"  # @param {type: "string"}
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "global")

from google import genai  # noqa: E402

client = genai.Client(
    vertexai=True, project=PROJECT_ID, location=LOCATION
)

### Import libraries

Import the Google Gen AI SDK types used for grounding configuration, along with utilities for REST API calls and display.

In [ ]:
import base64
import json
import re

import google.auth
import google.auth.transport.requests as auth_requests
import requests as http_requests
from IPython.core.display import Markdown
from IPython.display import Image as IPImage
from IPython.display import display
from google.genai.types import (
    GenerateContentConfig,
    GenerateContentResponse,
    GoogleMaps,
    GoogleSearch,
    LatLng,
    Part,
    Tool,
    ToolConfig,
    RetrievalConfig,
)

### Helper functions

`print_grounding_data()` renders a grounded response with inline citation markers and source links. `rest_endpoint()` builds the Gemini REST URL for features not yet in the SDK.

In [ ]:
def print_grounding_data(response: GenerateContentResponse):
    """Print response with grounding citations in Markdown."""
    candidate = (
        response.candidates[0] if response.candidates else None
    )
    metadata = getattr(candidate, "grounding_metadata", None)

    if not metadata:
        print("Response does not contain grounding metadata.")
        display(Markdown(response.text))
        return

    ENCODING = "utf-8"
    text_bytes = response.text.encode(ENCODING)
    parts = []
    last = 0

    for support in metadata.grounding_supports or []:
        end = support.segment.end_index
        parts.append(text_bytes[last:end].decode(ENCODING))
        indices = support.grounding_chunk_indices
        parts.append(
            " " + "".join(f"[{i + 1}]" for i in indices)
        )
        last = end

    parts.append(text_bytes[last:].decode(ENCODING))
    parts.append("\n\n----\n## Grounding Sources\n")

    if chunks := metadata.grounding_chunks:
        parts.append("### Grounding Chunks\n")
        for i, chunk in enumerate(chunks, 1):
            ctx = (
                chunk.web
                or chunk.retrieved_context
                or chunk.maps
            )
            if not ctx:
                continue
            uri = (ctx.uri or "").replace(
                "gs://",
                "https://storage.googleapis.com/",
                1,
            ).replace(" ", "%20")
            title = ctx.title or "Source"
            parts.append(f"{i}. [{title}]({uri})\n")
            if getattr(ctx, "place_id", None):
                pid = ctx.place_id
                parts.append(
                    f"    - Place ID: `{pid}`\n\n"
                )
            if getattr(ctx, "text", None):
                parts.append(f"{ctx.text}\n\n")

    if metadata.web_search_queries:
        queries = metadata.web_search_queries
        parts.append(f"\n**Web Search Queries:** {queries}\n")
        if metadata.search_entry_point:
            sep = metadata.search_entry_point
            parts.append(
                f"\n**Search Entry Point:**\n"
                f"{sep.rendered_content}\n"
            )
    elif metadata.retrieval_queries:
        queries = metadata.retrieval_queries
        parts.append(
            f"\n**Retrieval Queries:** {queries}\n"
        )

    widget_attr = "google_maps_widget_context_token"
    if hasattr(metadata, widget_attr):
        token = getattr(metadata, widget_attr)
        if token:
            parts.append(
                f"\n**Maps Widget Token:**"
                f" `{token[:50]}...`\n"
            )

    display(Markdown("".join(parts)))


def get_authorized_session():
    """Create an authorized HTTP session for REST."""
    creds = google.auth.default()[0]
    return auth_requests.AuthorizedSession(creds)


def rest_endpoint(model: str) -> str:
    """Build the Agent Platform generateContent REST URL."""
    return (
        "https://aiplatform.googleapis.com/v1"
        f"/projects/{PROJECT_ID}"
        "/locations/global/publishers/google"
        f"/models/{model}:generateContent"
    )

Initialize the Gemini model from Agent Platform:

In [ ]:
MODEL_ID = "gemini-2.5-flash"  # @param {type: "string"}

---
## 1. Grounding with Google Search

Google Search grounding lets Gemini issue web searches during generation and construct answers based on the results. The response includes **grounding metadata** with source citations: `grounding_chunks` (retrieved web pages), `grounding_supports` (text segments mapped to specific sources with confidence scores), and `web_search_queries` (the actual queries the model used). A `search_entry_point` widget is also returned for display per Terms of Service.

In the Concierge AI app, the `web_search` tool uses this to answer questions about current events, show schedules, ticket prices, and more.

### Without grounding

First, let's see what happens when we ask Gemini a time-sensitive question without grounding. The model can only rely on its training data, which may be outdated or incomplete.

In [ ]:
PROMPT = "What shows are playing in Las Vegas this week?"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
)

display(Markdown(response.text))

### With Google Search grounding

Add `GoogleSearch` as a tool to get grounded, up-to-date answers with source citations. The model autonomously decides which search queries to run, retrieves the results, and weaves them into the response with inline references.

In [ ]:
google_search_tool = Tool(google_search=GoogleSearch())

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(tools=[google_search_tool]),
)

print_grounding_data(response)

---
## 2. Grounding with Google Maps

Google Maps grounding connects Gemini to Google Maps Platform data for location-based queries. The model autonomously decides when to call Maps APIs based on geographic intent in the prompt. Set `enable_widget=True` to get an embeddable interactive map alongside the text response, and use `lat_lng` in `RetrievalConfig` to bias results toward a specific area. The grounding metadata includes `place_id` values that can be used with other Google Maps Platform APIs.

In the Concierge AI app, three tools use this:
- `web_search` — finds nearby places and returns a map widget
- `maps_route` — gets directions, distance, and duration
- `maps_search_along_route` — finds stops along a driving route

### Place search

Search for nearby places by passing a natural-language query with the `GoogleMaps` tool. The model returns place results along with an interactive map widget.

In [ ]:
google_maps_tool = Tool(google_maps=GoogleMaps(enable_widget=True))

PROMPT = "Find the best sushi restaurants near the Las Vegas Strip"

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(
        tools=[google_maps_tool],
        tool_config=ToolConfig(
            retrieval_config=RetrievalConfig(
                # Anchor "near me" queries to the Las Vegas Strip
                lat_lng=LatLng(latitude=36.1699, longitude=-115.1398)
            ),
        ),
    ),
)

print_grounding_data(response)

### Route — directions, distance, and duration

This is the pattern used by the `maps_route` tool. The model infers routing intent from a natural-language prompt when Google Maps grounding is enabled.

In [ ]:
from urllib.parse import quote_plus

ORIGIN = "3950 S Las Vegas Blvd, Las Vegas, NV 89119"
DESTINATION = "Red Rock Canyon"

PROMPT = (
    f"Give me driving directions from {ORIGIN} "
    f"to {DESTINATION}. "
    f"Include the total travel time and distance."
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(
        tools=[Tool(google_maps=GoogleMaps(enable_widget=True))],
    ),
)

print_grounding_data(response)

maps_url = (
    "https://www.google.com/maps/dir/?api=1"
    f"&origin={quote_plus(ORIGIN)}"
    f"&destination={quote_plus(DESTINATION)}"
    "&travelmode=driving"
)
display(Markdown(
    f"**[Open route in Google Maps]({maps_url})**"
))

### Search along route — POIs along a driving route

This is the pattern used by the `maps_search_along_route` tool. The model computes a route and finds matching POIs along it.

In [ ]:
ORIGIN = "3950 S Las Vegas Blvd, Las Vegas, NV 89119"
DESTINATION = "McCarran Airport"

PROMPT = (
    f"Find the top 3 coffee shops along the driving route "
    f"from {ORIGIN} to {DESTINATION}."
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents=PROMPT,
    config=GenerateContentConfig(
        tools=[Tool(google_maps=GoogleMaps(enable_widget=True))],
    ),
)

print_grounding_data(response)

# Extract place_ids from grounding chunks as waypoints
candidate = response.candidates[0] if response.candidates else None
gm = getattr(candidate, "grounding_metadata", None)
waypoints = []
if gm and gm.grounding_chunks:
    for chunk in gm.grounding_chunks:
        pid = getattr(chunk.maps, "place_id", None) if chunk.maps else None
        if pid:
            waypoints.append(f"place_id:{pid}")

maps_url = (
    "https://www.google.com/maps/dir/?api=1"
    f"&origin={quote_plus(ORIGIN)}"
    f"&destination={quote_plus(DESTINATION)}"
    "&travelmode=driving"
)
if waypoints:
    # Don't encode | and : — Maps URL API needs them raw
    maps_url += "&waypoints=" + "|".join(waypoints)

display(Markdown(
    f"**[Open route with stops in Google Maps]({maps_url})**"
))

---
## 3. Grounding with Vector Search 2.0

Vector Search 2.0 is a fully managed, AI-native search engine that evolves from the original index-as-a-service into a comprehensive storage and retrieval system. Instead of managing indexes as the primary resource, you work with **Collections** of **Data Objects** — a replicated, scalable storage engine that serves as a unified data source for AI applications.

A key feature is **auto-embeddings**: embeddings are generated automatically by [`gemini-embedding-2`](https://docs.cloud.google.com/gemini-enterprise-agent-platform/models/embeddings) via `vertex_embedding_config` in the vector schema. The `text_template` field controls what text gets embedded (e.g., `"{title} {category} {content}"`), and the `task_type` optimizes the embedding for different use cases — `RETRIEVAL_DOCUMENT` at indexing time and `QUESTION_ANSWERING` at query time.

In the Concierge AI app, the `vs2_search` tool queries a hotel knowledge base collection to answer guest questions about rooms, dining, spa, directions, policies, and more.

This section creates a small demo collection, adds hotel info items, and runs semantic search queries.

### Enable the Vector Search API

The Vector Search API must be enabled on your project before using this section.

In [ ]:
# noqa: E225
!gcloud services enable vectorsearch.googleapis.com --project={PROJECT_ID}

### Create a collection with auto-embeddings

Define data and vector schemas, then create a collection. The `vertex_embedding_config` in the vector schema tells Vector Search 2.0 which embedding model to use and how to construct the text input for embedding.

In [ ]:
from google.cloud import vectorsearch_v1beta

VS2_LOCATION = "us-central1"
COLLECTION_ID = "notebook-hotel-demo"  # @param {type: "string"}
VS2_PARENT = f"projects/{PROJECT_ID}/locations/{VS2_LOCATION}"
COLLECTION_PATH = f"{VS2_PARENT}/collections/{COLLECTION_ID}"

vs_client = vectorsearch_v1beta.VectorSearchServiceClient()
data_client = vectorsearch_v1beta.DataObjectServiceClient()
search_client = vectorsearch_v1beta.DataObjectSearchServiceClient()

# Data schema: fields stored alongside each vector
DATA_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "content": {"type": "string"},
        "category": {"type": "string"},
    },
}

# Vector schema: auto-embedding config — VS2 calls gemini-embedding-2
# automatically using the text_template to build the input text
VECTOR_SCHEMA = {
    "content_embedding": {
        "dense_vector": {
            "dimensions": 768,
            "vertex_embedding_config": {
                "model_id": "gemini-embedding-2",
                "text_template": "{title} {category} {content}",
                "task_type": "RETRIEVAL_DOCUMENT",
            },
        },
    },
}

# Create the collection (idempotent — skips if it already exists)
try:
    vs_client.get_collection(
        request=vectorsearch_v1beta.GetCollectionRequest(name=COLLECTION_PATH)
    )
    print(f"Collection '{COLLECTION_ID}' already exists")
except Exception:
    print(f"Creating collection '{COLLECTION_ID}'...")
    op = vs_client.create_collection(
        request=vectorsearch_v1beta.CreateCollectionRequest(
            parent=VS2_PARENT,
            collection_id=COLLECTION_ID,
            collection={
                "data_schema": DATA_SCHEMA,
                "vector_schema": VECTOR_SCHEMA,
            },
        )
    )
    op.result()
    print(f"Collection '{COLLECTION_ID}' created")

### Add hotel knowledge base items

Each item has a `title`, `content`, and `category`. The `vectors` field is left empty — Vector Search 2.0 generates embeddings automatically using the `vertex_embedding_config` defined in the schema.

In [ ]:
HOTEL_INFO = [
    {
        "id": "room-01",
        "title": "Deluxe Room",
        "content": (
            "45 sqm room with king or twin beds, "
            "floor-to-ceiling windows overlooking the "
            "Imperial Palace gardens. Features include "
            "Simmons Beautyrest mattress, Egyptian cotton "
            "400-thread-count linens, 55-inch OLED TV, "
            "Nespresso machine, minibar, marble bathroom "
            "with rain shower and deep soaking tub, "
            "heated toilet seat, and complimentary Aesop "
            "amenities. Rate: from $450 per night."
        ),
        "category": "rooms",
    },
    {
        "id": "room-02",
        "title": "Premier Suite",
        "content": (
            "85 sqm suite with separate living area, "
            "dining table for 4, and panoramic Tokyo "
            "skyline views. Includes all Deluxe Room "
            "amenities plus Bang & Olufsen sound system, "
            "walk-in closet, dual vanity bathroom, and "
            "access to the Executive Lounge. Turndown "
            "service with Japanese wagashi sweets. "
            "Rate: from $1,000 per night."
        ),
        "category": "rooms",
    },
    {
        "id": "dine-01",
        "title": "Sakura Restaurant",
        "content": (
            "Our signature 2-Michelin-star kaiseki "
            "restaurant on the 36th floor, led by Chef "
            "Takeshi Yamamoto. Multi-course seasonal "
            "tasting menus featuring the finest "
            "Tsukiji-sourced seafood, Wagyu beef, and "
            "Kyoto vegetables. Omakase dinner: $250. "
            "Open for dinner 5:30pm-9:30pm (last "
            "seating). Reservations essential. "
            "Smart casual dress code."
        ),
        "category": "dining",
    },
    {
        "id": "dine-02",
        "title": "The Brasserie — All-Day Dining",
        "content": (
            "International restaurant on the lobby "
            "level serving breakfast buffet "
            "(6:30am-10:30am, $40), lunch a la carte "
            "(11:30am-2:30pm), and dinner "
            "(5:30pm-10pm). Highlights include the "
            "fresh sashimi station at breakfast, "
            "hand-rolled pasta, and Sunday champagne "
            "brunch ($85 with free-flow Veuve "
            "Clicquot). Children's menu available."
        ),
        "category": "dining",
    },
    {
        "id": "spa-01",
        "title": "Spa Overview and Treatments",
        "content": (
            "The Serenity Spa occupies the entire 5th "
            "floor with 12 treatment rooms, including "
            "3 couples' suites. Signature treatment: "
            "'Sakura Renewal' — 90-minute hot stone "
            "massage with cherry blossom oil ($195). "
            "Other popular treatments: Shiatsu massage "
            "(60 min, $125), Hinoki Facial (75 min, "
            "$155), and Matcha Body Scrub (45 min, "
            "$100). Open 9am-9pm daily."
        ),
        "category": "spa",
    },
    {
        "id": "pol-01",
        "title": "Check-in and Check-out Times",
        "content": (
            "Standard check-in: 3:00pm. Standard "
            "check-out: 12:00pm (noon). Early check-in "
            "from 1:00pm available for $50 (subject to "
            "availability). Late check-out until 3:00pm: "
            "$55; until 6:00pm: half-day rate. Luggage "
            "storage is complimentary before check-in "
            "and after check-out."
        ),
        "category": "policies",
    },
    {
        "id": "dir-01",
        "title": "Getting to the Hotel from Narita",
        "content": (
            "From Narita International Airport, take "
            "the Narita Express (N'EX) to Tokyo Station "
            "(approx. 60 minutes), then transfer to the "
            "Marunouchi Line to Otemachi Station "
            "(2 minutes). Exit C6b leads directly to "
            "the hotel lobby. Alternatively, our airport "
            "limousine bus departs every 30 minutes. "
            "Private car transfer: $250."
        ),
        "category": "directions",
    },
    {
        "id": "fit-01",
        "title": "Fitness Center",
        "content": (
            "24-hour fitness center on the 5th floor "
            "featuring Technogym equipment: 15 cardio "
            "machines (treadmills, bikes, ellipticals, "
            "rowing machines), full free weight section "
            "(dumbbells up to 50kg, squat rack, bench "
            "press), TRX suspension training, and "
            "stretching area with yoga mats. Personal "
            "training sessions available ($85/hour). "
            "Complimentary workout attire and towels."
        ),
        "category": "fitness",
    },
    {
        "id": "act-01",
        "title": "Cultural Experiences at the Hotel",
        "content": (
            "Daily cultural activities for guests "
            "(free of charge): Morning meditation in "
            "the zen garden (6:30am), origami workshop "
            "(10am, lobby lounge), Japanese calligraphy "
            "class (2pm, Culture Room), ikebana flower "
            "arrangement (4pm, Saturdays). Premium "
            "experiences: private tea ceremony with a "
            "tea master ($100/person, 60 min), sake "
            "tasting with sommelier ($85/person)."
        ),
        "category": "activities",
    },
    {
        "id": "wifi-01",
        "title": "WiFi and Internet Access",
        "content": (
            "Complimentary high-speed WiFi available "
            "throughout the hotel (lobby, rooms, "
            "restaurants, pool, spa). Network name: "
            "'GrandSapphire-Guest'. Connect and enter "
            "your room number and last name. Speed: "
            "200 Mbps download / 100 Mbps upload. "
            "Wired Ethernet connections available in "
            "all rooms. IT support: dial ext. 9 (24/7)."
        ),
        "category": "wifi",
    },
]

# Leave vectors empty — VS2 auto-generates them via vertex_embedding_config
batch_request = [
    {
        "data_object_id": item["id"],
        "data_object": {
            "data": {
                "title": item["title"],
                "content": item["content"],
                "category": item["category"],
            },
            "vectors": {},
        },
    }
    for item in HOTEL_INFO
]

try:
    data_client.batch_create_data_objects(
        request=(
            vectorsearch_v1beta
            .BatchCreateDataObjectsRequest(
                parent=COLLECTION_PATH,
                requests=batch_request,
            )
        )
    )
    print(f"Created {len(HOTEL_INFO)} data objects")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Data objects already exist (skipping)")
    else:
        raise

### Semantic search

Search the hotel knowledge base using natural-language queries. The `task_type="QUESTION_ANSWERING"` parameter tells the embedding model to optimize for question-answering retrieval.

In [ ]:
import time

# Allow a moment for auto-embeddings to be generated
time.sleep(3)


def vs2_search(query, category="", top_k=3):
    """Search the hotel knowledge base."""
    filter_kwargs = {}
    if category:
        filter_kwargs["filter"] = {
            "category": {"$eq": category}
        }

    request = vectorsearch_v1beta.SearchDataObjectsRequest(
        parent=COLLECTION_PATH,
        semantic_search=vectorsearch_v1beta.SemanticSearch(
            search_text=query,
            search_field="content_embedding",
            # QUESTION_ANSWERING optimizes query-side embedding
            # (vs RETRIEVAL_DOCUMENT used at indexing time)
            task_type="QUESTION_ANSWERING",
            top_k=top_k,
            output_fields=(
                vectorsearch_v1beta.OutputFields(
                    data_fields=["*"],
                )
            ),
            **filter_kwargs,
        ),
    )
    results = search_client.search_data_objects(request)

    parts = [f"### Results for: *{query}*\n"]
    if category:
        parts.append(
            f"**Filter:** category = `{category}`\n\n"
        )
    parts.append("\n")

    count = 0
    for item in results:
        data = item.data_object.data
        title = data.get("title", "")
        cat = data.get("category", "")
        content = data.get("content", "")[:200]
        parts.append(f"**{title}** (`{cat}`)<br>\n")
        parts.append(f"{content}...\n\n")
        count += 1

    if count == 0:
        parts.append("*No results found.*\n")

    display(Markdown("".join(parts)))


vs2_search("What time is checkout?")

In [ ]:
vs2_search("How do I get to the hotel from the airport?")

### Filtered search

Vector Search 2.0 combines vector similarity with payload filtering in a single query. Narrow results by category using the MongoDB-style `$eq` filter operator.

In [ ]:
vs2_search("What's available?", category="dining")

In [ ]:
vs2_search("What activities can I do?", category="activities")

---
## 4. Image Search

Image Search via `enhancedContent.imageSearch` returns image thumbnails from Google Search results. In the Concierge AI app, the `web_search` tool uses this to show photos of places, attractions, and restaurants.

**Note:** This is a preview feature that requires allowlisting on your project. If your project does not have it enabled, the call will succeed but return 0 image attachments (falling back to regular Google Search grounding). The REST API is used because the SDK does not yet support `enhancedContent.imageSearch` on Agent Platform.

In [ ]:
# REST API — the SDK does not yet support enhancedContent.imageSearch
payload = json.dumps({
    "contents": [{
        "role": "user",
        "parts": [{
            "text": "Las Vegas Strip at night photos",
        }],
    }],
    "tools": [{
        "googleSearch": {
            "enhancedContent": {"imageSearch": {}},
        },
    }],
}).encode()

resp = get_authorized_session().post(
    rest_endpoint(MODEL_ID),
    data=payload,
    headers={"Content-Type": "application/json"},
    timeout=30,
)
resp.raise_for_status()
data = resp.json()

candidate = data.get("candidates", [{}])[0]
gm = candidate.get("groundingMetadata", {})

# Image thumbnails are returned in the attachments field
parts = ["### Image Search Results\n\n"]
for att in gm.get("attachments", [])[:5]:
    img = att.get("image", {})
    if not img:
        continue
    source = img.get("source", {})
    thumbnail = img.get("thumbnail", {})
    title = source.get("title", "Untitled")
    uri = source.get("uri", "")
    parts.append(f"**{title}**\n")
    parts.append(f"- Source page: {uri}\n")
    thumb_uri = thumbnail.get("uri")
    if thumb_uri:
        parts.append(f"- Thumbnail: ![]({thumb_uri})\n")
    parts.append("\n")

display(Markdown("".join(parts)))

---
## 5. Video Search (YouTube)

Video search uses Google Search grounding with a video-biased query to find YouTube content. In the Concierge AI app, the `web_search` tool runs this in parallel with text, image, and maps searches.

YouTube video IDs are extracted from grounding chunks by following redirect URLs.

In [ ]:
# Grounding chunks contain Google redirect URLs, not direct YouTube links.
# Follow the redirect to extract the actual video ID.
YT_VIDEO_RE = re.compile(
    r"(?:youtube\.com/"
    r"(?:watch\?.*v=|shorts/|embed/|live/|v/)"
    r"|youtu\.be/)"
    r"([A-Za-z0-9_-]{11})"
)


def resolve_youtube_id(redirect_uri):
    """Follow redirect URL, extract YouTube video ID."""
    try:
        resp = http_requests.head(
            redirect_uri, allow_redirects=True, timeout=5,
        )
        m = YT_VIDEO_RE.search(resp.url)
        return m.group(1) if m else ""
    except Exception:
        return ""


# Use REST to bias the query toward video results
payload = json.dumps({
    "contents": [{
        "role": "user",
        "parts": [{
            "text": (
                "Bellagio Fountains Las Vegas "
                "youtube video"
            ),
        }],
    }],
    "tools": [{"googleSearch": {}}],
}).encode()

resp = get_authorized_session().post(
    rest_endpoint(MODEL_ID),
    data=payload,
    headers={"Content-Type": "application/json"},
    timeout=30,
)
resp.raise_for_status()
data = resp.json()

candidate = data.get("candidates", [{}])[0]
gm = candidate.get("groundingMetadata", {})

parts = ["### Video Search Results\n\n"]
found = 0
for chunk in gm.get("groundingChunks", []):
    web = chunk.get("web", {})
    if not web:
        continue
    uri = web.get("uri", "")
    title = web.get("title", "")
    vid = resolve_youtube_id(uri)
    if vid:
        parts.append(f"**{title}**\n")
        parts.append(f"- Video ID: `{vid}`\n")
        yt_url = f"https://www.youtube.com/watch?v={vid}"
        parts.append(f"- URL: {yt_url}\n")
        thumb = (
            "https://i.ytimg.com"
            f"/vi/{vid}/hqdefault.jpg"
        )
        parts.append(f"- Thumbnail: ![]({thumb})\n\n")
        found += 1
    if found >= 3:
        break

if found == 0:
    parts.append(
        "*No YouTube videos found in grounding "
        "chunks. Web results:*\n\n"
    )
    for chunk in gm.get("groundingChunks", [])[:5]:
        web = chunk.get("web", {})
        if web:
            t = web.get("title", "")
            u = web.get("uri", "")
            parts.append(f"- [{t}]({u})\n")

display(Markdown("".join(parts)))

---
## 6. Image Search for Image Generation

This is a distinct feature from the thumbnail Image Search in section 4. Here, `googleSearch.searchTypes.imageSearch` lets the image generation model use real Google Image Search results as **visual context** during image generation.

In the Concierge AI app, `request_postcard` uses this to generate photorealistic selfie postcards of venues, grounded in actual web photos of those places.

**Important:** The prompt must explicitly invite the search (e.g., "Search Google Images for photos of X, then generate..."). Without this, the model generates an image with empty `groundingMetadata`.

**Note:** This uses `gemini-3.1-flash-image-preview`, a preview model.

In [ ]:
# This model supports native image generation with search grounding
IMAGE_GEN_MODEL = "gemini-3.1-flash-image-preview"

# REST API — the SDK does not yet support searchTypes.imageSearch
payload = json.dumps({
    "contents": [{
        "role": "user",
        "parts": [{
            "text": (
                "Search Google Images for photos of "
                "the Bellagio Fountains in Las Vegas, "
                "then use those reference photos to "
                "generate a beautiful watercolor "
                "illustration of the fountains at "
                "sunset with vivid colors."
            ),
        }],
    }],
    "tools": [{
        "googleSearch": {
            # imageSearch provides visual context for generation;
            # webSearch adds text context from web pages
            "searchTypes": {
                "imageSearch": {},
                "webSearch": {},
            },
        },
    }],
    "generationConfig": {
        # Include IMAGE to enable native image generation
        "responseModalities": ["TEXT", "IMAGE"],
    },
}).encode()

resp = get_authorized_session().post(
    rest_endpoint(IMAGE_GEN_MODEL),
    data=payload,
    headers={"Content-Type": "application/json"},
    timeout=120,
)
resp.raise_for_status()
data = resp.json()

candidate = data.get("candidates", [{}])[0]

# Display generated image
for part in candidate.get("content", {}).get("parts", []):
    inline = (
        part.get("inlineData")
        or part.get("inline_data")
    )
    if inline and inline.get("data"):
        mime = (
            inline.get("mimeType")
            or inline.get("mime_type")
            or "image/png"
        )
        fmt = mime.split("/")[-1]
        display(IPImage(
            data=base64.b64decode(inline["data"]),
            format=fmt,
        ))
    if "text" in part:
        display(Markdown(part["text"]))

# Display image search sources (attribution)
gm = candidate.get("groundingMetadata", {})
sources = []
for chunk in gm.get("groundingChunks", []):
    img_chunk = chunk.get("image") or {}
    web_chunk = chunk.get("web") or {}
    uri = (
        img_chunk.get("sourceUri")
        or web_chunk.get("uri")
        or ""
    )
    raw_title = (
        img_chunk.get("title")
        or web_chunk.get("title")
        or ""
    )
    title = re.sub(r"</?b>", "", raw_title).strip()
    if uri:
        sources.append(f"- [{title or uri}]({uri})")

if sources:
    display(Markdown(
        "### Image Sources (attribution)\n\n"
        + "\n".join(sources[:5])
    ))
else:
    print(
        "No image search sources returned "
        "(the model may not have triggered "
        "the search)."
    )

---
## 7. Build an ADK Concierge Agent

Now we bring everything together using [Agent Development Kit (ADK)](https://google.github.io/adk-docs/) to create an AI agent that combines all the grounding methods. The agent autonomously decides which tool to call based on the user's question:

- **`GoogleSearchTool`** (built-in) — the model automatically grounds its response in Google Search results. Unlike custom tools, this requires no sidecar call — the model invokes Google Search internally during generation and returns grounded text with citations.
- **`hotel_search`** — queries the Vector Search 2.0 hotel knowledge base (rooms, dining, spa, policies, etc.)
- **`maps_search`** — finds nearby places via Google Maps grounding
- **`maps_route`** — gets directions, distance, and duration between two locations via Google Maps grounding

The custom tools use the **sidecar pattern**: each makes a separate `generateContent` call with the appropriate grounding tool (`GoogleMaps`) or calls the Vector Search 2.0 API directly, then returns structured results. The ADK agent's main model orchestrates which tools to call and synthesizes the results into a natural response.

### Define grounding tools

ADK provides a built-in `GoogleSearchTool` that the model invokes directly during generation — no sidecar API call needed. The model's response is automatically grounded in Google Search results with inline citations. Set `bypass_multi_tools_limit=True` to use it alongside custom function tools.

For Maps and Vector Search, we define custom function tools that make sidecar `generateContent` or Vector Search API calls and return structured results. ADK extracts the tool name, description, and parameter schemas from the function's docstring and type annotations (Google-style `Args:` / `Returns:` format).

In [ ]:
from google.adk.tools import GoogleSearchTool

google_search_tool = GoogleSearchTool(
    bypass_multi_tools_limit=True,
)


def maps_search(query: str) -> dict:
    """Search Google Maps for nearby places.

    Use this tool to find restaurants, attractions, or
    other points of interest near Las Vegas.

    Args:
        query: What to search for, e.g. "sushi restaurants
            near the Las Vegas Strip".

    Returns:
        A dict with summary text and a list of places.
    """
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=query,
        config=GenerateContentConfig(
            tools=[Tool(
                google_maps=GoogleMaps(enable_widget=True)
            )],
            tool_config=ToolConfig(
                retrieval_config=RetrievalConfig(
                    lat_lng=LatLng(
                        latitude=36.1699,
                        longitude=-115.1398,
                    )
                ),
            ),
        ),
    )
    candidate = response.candidates[0] if response.candidates else None
    gm = getattr(candidate, "grounding_metadata", None)

    places = []
    if gm and gm.grounding_chunks:
        for chunk in gm.grounding_chunks:
            if chunk.maps:
                places.append({
                    "title": chunk.maps.title or "",
                    "uri": chunk.maps.uri or "",
                    "place_id": getattr(
                        chunk.maps, "place_id", ""
                    ),
                })

    return {
        "query": query,
        "summary": response.text or "",
        "places": places[:5],
    }


def maps_route(
    origin: str,
    destination: str,
    travel_mode: str = "driving",
) -> dict:
    """Get directions, distance, and duration.

    Use this tool when a guest asks how to get somewhere,
    travel time, or directions between two places.

    Args:
        origin: Starting address — use a street address,
            e.g. "3950 S Las Vegas Blvd, Las Vegas, NV".
        destination: Ending location, e.g. "Red Rock Canyon".
        travel_mode: One of "driving", "walking", "bicycling",
            "transit". Defaults to "driving".

    Returns:
        A dict with route summary and waypoint places.
    """
    prompt = (
        f"Give me {travel_mode} directions from "
        f"{origin} to {destination}. "
        f"Include the total travel time and distance."
    )
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=GenerateContentConfig(
            tools=[Tool(
                google_maps=GoogleMaps(enable_widget=True)
            )],
        ),
    )
    candidate = response.candidates[0] if response.candidates else None
    gm = getattr(candidate, "grounding_metadata", None)

    places = []
    if gm and gm.grounding_chunks:
        for chunk in gm.grounding_chunks:
            if chunk.maps:
                places.append({
                    "title": chunk.maps.title or "",
                    "uri": chunk.maps.uri or "",
                    "place_id": getattr(
                        chunk.maps, "place_id", ""
                    ),
                })

    return {
        "origin": origin,
        "destination": destination,
        "travel_mode": travel_mode,
        "summary": response.text or "",
        "places": places[:5],
    }


def hotel_search(query: str, category: str = "") -> dict:
    """Search the hotel knowledge base.

    Use this tool to answer guest questions about the hotel
    — rooms, dining, spa, fitness, check-in/out, Wi-Fi,
    activities, directions from the airport, and policies.

    Args:
        query: Natural-language question, e.g. "what time
            is checkout" or "tell me about the spa".
        category: Optional filter to narrow results. One of:
            "rooms", "dining", "spa", "fitness", "activities",
            "policies", "directions", "wifi".

    Returns:
        A dict with matching hotel information items.
    """
    filter_kwargs = {}
    if category:
        filter_kwargs["filter"] = {
            "category": {"$eq": category}
        }

    request = vectorsearch_v1beta.SearchDataObjectsRequest(
        parent=COLLECTION_PATH,
        semantic_search=vectorsearch_v1beta.SemanticSearch(
            search_text=query,
            search_field="content_embedding",
            task_type="QUESTION_ANSWERING",
            top_k=3,
            output_fields=(
                vectorsearch_v1beta.OutputFields(
                    data_fields=["*"],
                )
            ),
            **filter_kwargs,
        ),
    )
    results = search_client.search_data_objects(request)

    items = []
    for item in results:
        data = item.data_object.data
        items.append({
            "title": data.get("title", ""),
            "content": data.get("content", "")[:300],
            "category": data.get("category", ""),
        })

    return {"query": query, "items": items}


print("Tools: google_search_tool (built-in), maps_search, maps_route, hotel_search")

### Create the ADK Concierge Agent

The agent's instruction teaches it when to use each tool. The `InMemoryRunner` provides a simple way to test the agent in a notebook without setting up a server.

In [ ]:
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

# Use Vertex AI auth (not API key) for the ADK agent
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = "global"

AGENT_INSTRUCTION = """
You are a concierge for a luxury hotel in Las Vegas.
The hotel is located at 3950 S Las Vegas Blvd, Las Vegas, NV 89119.

You have access to Google Search (built-in) and three custom tools.
Pick the right one based on the guest's question:

1. **Google Search** (built-in) — automatically grounds your response
   in web search results for current events, show schedules, ticket
   prices, and anything outside the hotel's own knowledge base. You
   don't need to call it explicitly — just answer and the model will
   search the web as needed.

2. `hotel_search` — for questions about the hotel itself (rooms, dining,
   spa, fitness, check-in/out, Wi-Fi, activities, directions from the
   airport). Always try this first for hotel-related questions. Use the
   optional `category` parameter to narrow results.

3. `maps_search` — for finding nearby places and points of interest
   around Las Vegas.

4. `maps_route` — for getting directions between two locations with
   travel time and distance. Use the hotel address as the origin
   when the guest asks "how do I get to X" without specifying a
   starting point.

Always answer in a friendly, helpful tone. Include specific details
(prices, times, distances) from the tool results.
"""

concierge_agent = Agent(
    model="gemini-2.5-flash",
    name="concierge_agent",
    instruction=AGENT_INSTRUCTION,
    tools=[
        google_search_tool,
        hotel_search,
        maps_search,
        maps_route,
    ],
)

runner = InMemoryRunner(
    agent=concierge_agent, app_name="concierge_agent"
)

print("Concierge Agent initialized and ready!")

### Test the Agent

Let's test the agent with different types of questions. The agent should pick the right tool for each query and synthesize the results.

#### Test 1: Hotel knowledge (Vector Search 2.0)

The agent should call `hotel_search` to look up check-out policy from the hotel knowledge base.

In [ ]:
await runner.run_debug("What time is checkout and can I get a late checkout?")

#### Test 2: Current events (Google Search — built-in)

The agent uses the built-in `GoogleSearchTool` to automatically ground its response in web search results. No explicit tool call is needed — the model searches the web internally during generation.

In [ ]:
await runner.run_debug("What shows are playing in Las Vegas this week?")

#### Test 3: Nearby places (Google Maps)

The agent should call `maps_search` to find restaurants using Google Maps grounding with location context.

In [ ]:
await runner.run_debug("Find good sushi restaurants near the hotel")

#### Test 4: Directions (Google Maps route)

The agent should call `maps_route` with the hotel address as origin and Red Rock Canyon as destination, returning directions with travel time and distance.

In [ ]:
await runner.run_debug("How do I get to Red Rock Canyon from the hotel?")

---
## Cleaning up

Delete the Vector Search 2.0 collection created in this notebook.

In [ ]:
# Delete the demo collection
try:
    item_ids = [item["id"] for item in HOTEL_INFO]
    reqs = [
        vectorsearch_v1beta.DeleteDataObjectRequest(
            name=(
                f"{COLLECTION_PATH}"
                f"/dataObjects/{obj_id}"
            )
        )
        for obj_id in item_ids
    ]
    data_client.batch_delete_data_objects(
        request=(
            vectorsearch_v1beta
            .BatchDeleteDataObjectsRequest(
                parent=COLLECTION_PATH,
                requests=reqs,
            )
        )
    )
    print(f"Deleted {len(item_ids)} data objects")

    op = vs_client.delete_collection(
        request=(
            vectorsearch_v1beta
            .DeleteCollectionRequest(
                name=COLLECTION_PATH,
            )
        )
    )
    op.result(timeout=300)
    print(f"Collection '{COLLECTION_ID}' deleted")
except Exception as e:
    print(f"Cleanup error: {e}")

To avoid ongoing charges:

1. Delete your Google Cloud project if you no longer need it
2. Or disable the [Agent Platform](https://console.cloud.google.com/apis/api/aiplatform.googleapis.com) and [Vector Search](https://console.cloud.google.com/apis/api/vectorsearch.googleapis.com) APIs